# Field Scan : choix et test du détecteur réentraîné (reprise)

L'entraînement (12 époques, 6 points de contrôle) s'est terminé dans l'exécution précédente, mais le carnet a perdu le contact avec Kaggle juste après, avant le choix et le test. Ce carnet ne réentraîne rien : il applique la **règle fixée à l'avance** aux points de contrôle sauvegardés, puis regarde le test une fois.

**Avant de lancer** : Input = `field-scan-comptage` **et** la sortie du carnet d'entraînement (dossier `modeles_comptage/points_controle`) ; GPU ; Internet ; Save & Run All. Durée : environ 20 minutes.

## 1. Données

In [ ]:
import os, glob, zipfile, shutil
# Kaggle décompresse les zips au téléversement : chaque source est soit un zip, soit un dossier déjà décompressé,
# reconnu à un fichier témoin (val.csv pour ReforesTree, val_dense.csv pour Roboflow).
def localiser(noms_zip, temoin):
    for n in noms_zip:
        z = glob.glob(f'/kaggle/input/**/{n}', recursive=True)
        if z and os.path.isfile(z[0]):
            return 'zip', z[0]
    d = [os.path.dirname(p) for p in glob.glob(f'/kaggle/input/**/{temoin}', recursive=True)]
    assert d, f'Ni zip ({noms_zip}) ni dossier contenant {temoin} dans /kaggle/input : ajoute le jeu field-scan-comptage'
    return 'dossier', d[0]

SOURCES = {'rt': localiser(['deepforest_kaggle.zip', 'deepforest.zip'], 'val.csv'),
           'rf': localiser(['deepforest_roboflow.zip'], 'val_dense.csv')}
ckpts = glob.glob('/kaggle/input/**/bananiers_deepforest_v3_40.ckpt', recursive=True)
assert ckpts, 'Introuvable : bananiers_deepforest_v3_40.ckpt'
CKPT = ckpts[0]
print(SOURCES, '|', CKPT)

DONNEES = '/tmp/donnees'
SORTIE = '/kaggle/working/modeles_comptage'
os.makedirs(SORTIE, exist_ok=True); shutil.rmtree(DONNEES, ignore_errors=True)
os.makedirs(f'{DONNEES}/images')
for sous, (genre, chemin) in SOURCES.items():
    os.makedirs(f'{DONNEES}/{sous}')
    if genre == 'zip':
        with zipfile.ZipFile(chemin) as z:
            for m in z.infolist():
                c_ = m.filename.replace('\\', '/')          # zips PowerShell : « \ » -> « / »
                if c_.endswith('/'):
                    continue
                cible = (f'{DONNEES}/images/{os.path.basename(c_)}' if '/images/' in '/' + c_
                         else f'{DONNEES}/{sous}/{os.path.basename(c_)}')
                with z.open(m) as src, open(cible, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    else:
        for f in glob.glob(f'{chemin}/*.csv') + glob.glob(f'{chemin}/*.json'):
            shutil.copy(f, f'{DONNEES}/{sous}/')
        for f in glob.glob(f'{chemin}/**/images/*.png', recursive=True):
            os.symlink(f, f'{DONNEES}/images/{os.path.basename(f)}')   # lien : pas de copie de 1,5 Go
print(len(os.listdir(f'{DONNEES}/images')), 'tuiles ;', sorted(os.listdir(f'{DONNEES}/rt')), sorted(os.listdir(f'{DONNEES}/rf')))

## 2. Installation

In [ ]:
!pip -q install deepforest==2.1.0
import torch, deepforest, json, platform
from importlib.metadata import version
VERSIONS = {'deepforest': version('deepforest'), 'torch': torch.__version__, 'python': platform.python_version(),
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun'}
print(VERSIONS)
assert torch.cuda.is_available(), 'Pas de GPU : Settings > Accelerator > GPU'

## 3. Annotations

In [ ]:
import numpy as np, shutil
import pandas as pd
def lire(chemin):
    d = pd.read_csv(chemin); d['label'] = 'Tree'; return d
ENS = {'train_rt': lire(f'{DONNEES}/rt/train.csv'), 'val_rt': lire(f'{DONNEES}/rt/val.csv'),
       'train_rf': lire(f'{DONNEES}/rf/train_roboflow.csv'), 'val_dense': lire(f'{DONNEES}/rf/val_dense.csv'),
       'test_dense': lire(f'{DONNEES}/rf/test_dense.csv')}
for nom, d in ENS.items():
    d.to_csv(f'{DONNEES}/{nom}.csv', index=False)
    b = d[d.xmax > 0]
    print(f'{nom:10s} : {d.image_path.nunique():5d} tuiles, {len(b):6d} bananiers, côté médian {(b.xmax - b.xmin).median():.0f} px')
# Équilibre (règle fixée à l'avance) : les tuiles ReforesTree sont dupliquées, sous de nouveaux noms (sinon
# DeepForest regroupe les lignes d'une même image), jusqu'à représenter au moins 25 % des tuiles d'entraînement.
rt, rf = ENS['train_rt'][ENS['train_rt'].xmax > 0], ENS['train_rf'][ENS['train_rf'].xmax > 0]
# Volume limité (règle fixée après l'échec de la première exécution : plus de 1 h 30 par époque) :
# 1 500 tuiles Roboflow tirées au hasard, graine 2026.
MAX_TUILES_RF = 1500
tuiles_rf = rf.image_path.drop_duplicates().sample(frac=1, random_state=2026)
rf = rf[rf.image_path.isin(set(tuiles_rf.iloc[:MAX_TUILES_RF]))]
n_rt, n_rf = rt.image_path.nunique(), rf.image_path.nunique()
REPETITIONS = max(1, int(np.ceil(0.25 * n_rf / (0.75 * n_rt))))
copies = []
for k in range(REPETITIONS):
    c_ = rt.copy()
    if k > 0:
        c_['image_path'] = c_.image_path.str.replace('.png', f'_rep{k}.png', regex=False)
        for ancien, nouveau in zip(rt.image_path.unique(), c_.image_path.unique()):
            shutil.copy(f'{DONNEES}/images/{ancien}', f'{DONNEES}/images/{nouveau}')
    copies.append(c_)
train = pd.concat(copies + [rf])
train.to_csv(f'{DONNEES}/train_sans_vides.csv', index=False)
print(f'ReforesTree : {n_rt} tuiles x {REPETITIONS} ; Roboflow : {n_rf} tuiles ; '
      f'part ReforesTree {n_rt * REPETITIONS / (n_rt * REPETITIONS + n_rf):.0%}')
ENS['val_rt'][ENS['val_rt'].xmax > 0].to_csv(f'{DONNEES}/val_rt_sans_vides.csv', index=False)
print(f'Entraînement : {train[train.xmax > 0].image_path.nunique()} tuiles avec bananiers')

## 4. Mesure et modèle de départ

In [ ]:
import re, os, shutil
import numpy as np
from PIL import Image
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from deepforest import main as df_main

RAYON_PX = 10
SEUILS = (0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5)

def evaluer_seuils(m, ensemble, seuils=SEUILS, rayon=RAYON_PX):
    ref = pd.read_csv(f'{DONNEES}/{ensemble}.csv')
    m.model.eval(); ancien = getattr(m.model, 'score_thresh', None); m.model.score_thresh = 0.0
    preds = {}
    for nom in ref.image_path.unique():
        img = np.array(Image.open(f'{DONNEES}/images/{nom}').convert('RGB')).astype('float32') / 255
        with torch.no_grad():
            o = m.model([torch.tensor(img).permute(2, 0, 1).to(m.device)])[0]
        b, s = o['boxes'].cpu().numpy(), o['scores'].cpu().numpy()
        preds[nom] = (np.c_[(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2], s)
    m.model.score_thresh = ancien
    lignes = []
    for t in seuils:
        vp = fp = fn = 0
        for nom, r in ref.groupby('image_path'):
            r = r[r.xmax > 0]
            ref_xy = np.c_[(r.xmin + r.xmax) / 2, (r.ymin + r.ymax) / 2]
            det_xy = preds[nom][0][preds[nom][1] > t]
            n = 0
            if len(det_xy) and len(ref_xy):
                d = cdist(det_xy, ref_xy)
                li, co = linear_sum_assignment(np.where(d <= rayon, d, 1e6))
                n = int((d[li, co] <= rayon).sum())
            vp += n; fp += len(det_xy) - n; fn += len(ref_xy) - n
        p = vp / (vp + fp) if vp + fp else 0; r_ = vp / (vp + fn) if vp + fn else 0
        lignes.append(dict(seuil=t, reference=vp + fn, detections=vp + fp, vp=vp, precision=round(p, 3), rappel=round(r_, 3),
                           f1=round(2 * p * r_ / (p + r_), 3) if p + r_ else 0,
                           rapport_reel_sur_detecte=round((vp + fn) / max(vp + fp, 1), 3)))
    return pd.DataFrame(lignes)

# Le point de contrôle garde les chemins de Colab, où il a été entraîné ; DeepForest relit le CSV de validation
# en rechargeant le modèle. On recrée ces chemins avec nos propres fichiers (validation agroforestière).
COLAB = '/content/deepforest'
os.makedirs(COLAB, exist_ok=True)
for nom in ('val_tree_sans_vides.csv', 'val_tree.csv', 'train_tree_sans_vides.csv', 'train_tree.csv'):
    shutil.copy(f'{DONNEES}/val_rt_sans_vides.csv', f'{COLAB}/{nom}')
if not os.path.exists(f'{COLAB}/images'):
    os.symlink(f'{DONNEES}/images', f'{COLAB}/images')

depart = df_main.deepforest.load_from_checkpoint(CKPT).to('cuda')
for ens in ('val_rt', 'val_dense'):
    print(f'\nModèle de départ, {ens} :'); print(evaluer_seuils(depart, ens).to_string(index=False))
del depart; torch.cuda.empty_cache()

## 5. Choix de l'époque et du seuil (règle fixée à l'avance)

In [ ]:
REGLAGES = dict(depart='bananiers_deepforest_v3_40.ckpt', epochs=12, lr=0.0005, batch_size=2, ecretage_gradients=1.0, squelette_gele=True, tuiles_vides_retirees=True, repetitions_reforestree=34, part_minimale_reforestree=0.25, points_controle_toutes_les=2, remarque='entraînement fait dans la version précédente ; ce carnet ne fait que le choix et le test')
# Points de contrôle de l'exécution précédente (12 époques, un toutes les 2 époques), attachée en entrée
POINTS = sorted(glob.glob('/kaggle/input/**/points_controle/dense-epoch=*.ckpt', recursive=True))
assert POINTS, 'Attache en entrée la sortie du carnet d’entraînement (dossier modeles_comptage/points_controle)'
print([os.path.basename(f) for f in POINTS])
candidats = [(0, CKPT)] + [(int(re.search(r'epoch=(\d+)', f).group(1)) + 1, f) for f in POINTS]
lignes = []
for ep, f in candidats:
    m = df_main.deepforest.load_from_checkpoint(f).to('cuda')
    a, b = evaluer_seuils(m, 'val_rt'), evaluer_seuils(m, 'val_dense')
    for (_, ra), (_, rb) in zip(a.iterrows(), b.iterrows()):
        lignes.append(dict(epoque=ep, fichier=f, seuil=ra.seuil, f1_agroforesterie=ra.f1, f1_dense=rb.f1,
                           moyenne=round((ra.f1 + rb.f1) / 2, 4), rapport_dense=rb.rapport_reel_sur_detecte))
    del m; torch.cuda.empty_cache()
CHOIX = pd.DataFrame(lignes)
print(CHOIX.loc[CHOIX.groupby('epoque').moyenne.idxmax()].drop(columns='fichier').to_string(index=False))
meilleur = CHOIX.loc[CHOIX.moyenne.idxmax()]
print(f'\nRetenu : époque {int(meilleur.epoque)} (0 = modèle de départ), seuil {meilleur.seuil}, '
      f'F1 agroforesterie {meilleur.f1_agroforesterie}, F1 dense {meilleur.f1_dense}')
shutil.copy(meilleur.fichier, f'{SORTIE}/bananiers_deepforest_dense.ckpt')

## 6. Test (une seule fois)

In [ ]:
final = df_main.deepforest.load_from_checkpoint(f'{SORTIE}/bananiers_deepforest_dense.ckpt').to('cuda')
depart = df_main.deepforest.load_from_checkpoint(CKPT).to('cuda')
T_final, T_depart = evaluer_seuils(final, 'test_dense'), evaluer_seuils(depart, 'test_dense')
s = float(meilleur.seuil)
print('Test dense, modèle retenu (seuil retenu) :'); print(T_final[T_final.seuil == s].to_string(index=False))
print('\nTest dense, modèle de départ (son seuil de 0,4) :'); print(T_depart[T_depart.seuil == 0.4].to_string(index=False))
json.dump(dict(versions=VERSIONS, reglages=REGLAGES, resolution_cm=10, taille_tuile_px=400, pertes_par_epoque='non disponibles (journal de l’exécution d’entraînement interrompu)',
               choix=CHOIX.drop(columns='fichier').to_dict('records'), epoque_retenue=int(meilleur.epoque), seuil_retenu=s,
               f1_validation_agroforesterie=float(meilleur.f1_agroforesterie), f1_validation_dense=float(meilleur.f1_dense),
               test_dense_modele_retenu=T_final[T_final.seuil == s].to_dict('records'),
               test_dense_modele_depart_seuil_0_4=T_depart[T_depart.seuil == 0.4].to_dict('records'),
               regle='couple (époque, seuil) maximisant la moyenne des F1 de validation agroforesterie (Manuel Macias) et dense '
                     '(Count banana plants, valid) ; test regardé une fois, même région que Plants-bana (optimiste)'),
          open(f'{SORTIE}/entrainement_dense.json', 'w'), indent=2, ensure_ascii=False, default=float)
print(sorted(os.listdir(SORTIE)))